# C5 — Baseline Models

The C4 split is already frozen.

This notebook evaluates three fixed baselines:

1. Base-rate baseline
2. Logistic Regression
3. Gradient Boosting

No hyperparameter tuning is performed.

The test set is used only for final reporting, not for model selection.

All imputation and categorical encoding are fit inside each model pipeline
using the training partition only.

In [1]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "c4_splits").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLIT_DIR = PROJECT_ROOT / "data" / "c4_splits"
REPORT_DIR = PROJECT_ROOT / "reports" / "c5_baseline"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "y"

train = pd.read_csv(
    SPLIT_DIR / "train.csv",
    low_memory=False,
)
valid = pd.read_csv(
    SPLIT_DIR / "validation.csv",
    low_memory=False,
)
test = pd.read_csv(
    SPLIT_DIR / "test.csv",
    low_memory=False,
)

print("Train:", train.shape)
print("Validation:", valid.shape)
print("Test:", test.shape)


Train: (110385, 37)
Validation: (23471, 37)
Test: (23567, 37)


## 1. Feature Roles

In [2]:

ID_COLUMNS = [
    "customer_id",
    "subscription_id",
    "case_id",
]

POLICY_COLUMNS = [
    "logging_policy",
]

INTERVENTION_COLUMNS = [
    "action_type",
    "channel",
]

excluded = set(
    ID_COLUMNS + POLICY_COLUMNS + [TARGET]
)

features = [
    c for c in train.columns
    if c not in excluded
]

categorical_cols = [
    c for c in features
    if c in INTERVENTION_COLUMNS
    or not pd.api.types.is_numeric_dtype(train[c])
]

numeric_cols = [
    c for c in features
    if c not in categorical_cols
    and pd.api.types.is_numeric_dtype(train[c])
]

print("Numeric:", numeric_cols)
print("\nCategorical:", categorical_cols)
print("\nIntervention:",
      [c for c in INTERVENTION_COLUMNS if c in train.columns])


Numeric: ['amount_inr', 'customer_tenure_days', 'subscription_age_days', 'prior_cycles_seen', 'prior_successes', 'prior_success_rate', 'prior_failed_cases_before', 'prior_recovered_cases_before', 'retries_used_before', 'contacts_sent_before', 'hours_since_open', 'decision_seq', 'hour_of_day', 'day_of_month', 'hours_to_next_cluster_slot', 'salary_day_proxy_dom', 'instrument_fixed', 'day_of_week', 'in_salary_cluster', 'amount_inr_log1p']

Categorical: ['rail', 'bank_name', 'segment', 'plan_tier', 'value_tier', 'first_failure_class', 'failure_class_at_decision', 'age_band', 'state', 'locale_pref', 'action_type', 'channel']

Intervention: ['action_type', 'channel']


## 2. Evaluation Metrics

In [3]:

def evaluate(y_true, proba):
    proba = np.clip(
        np.asarray(proba, dtype=float),
        1e-7,
        1 - 1e-7,
    )
    y_true = np.asarray(y_true)

    return {
        "n_rows": len(y_true),
        "positive_rate": y_true.mean(),
        "roc_auc": roc_auc_score(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "log_loss": log_loss(y_true, proba),
        "brier": brier_score_loss(y_true, proba),
    }


## 3. Base-Rate Baseline

In [4]:

train_rate = float(train[TARGET].mean())

valid_proba_base = np.full(
    len(valid),
    train_rate,
)
test_proba_base = np.full(
    len(test),
    train_rate,
)

base_result = {
    "model": "base_rate",
    "validation": evaluate(
        valid[TARGET],
        valid_proba_base,
    ),
    "test": evaluate(
        test[TARGET],
        test_proba_base,
    ),
}

print(base_result)


{'model': 'base_rate', 'validation': {'n_rows': 23471, 'positive_rate': np.float64(0.22994333432746794), 'roc_auc': 0.5, 'pr_auc': 0.22994333432746794, 'log_loss': 0.5392162841828909, 'brier': 0.17707236922757918}, 'test': {'n_rows': 23567, 'positive_rate': np.float64(0.22590911019646115), 'roc_auc': 0.5, 'pr_auc': 0.22590911019646115, 'log_loss': 0.5343010146003823, 'brier': 0.17487952162976667}}


## 4. Logistic Regression

In [5]:

features = numeric_cols + categorical_cols

def make_preprocessor():
    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=2,
            ),
        ),
    ])

    return ColumnTransformer([
        (
            "numeric",
            numeric_pipeline,
            numeric_cols,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_cols,
        ),
    ])

logistic = Pipeline([
    ("preprocess", make_preprocessor()),
    (
        "model",
        LogisticRegression(
            solver="liblinear",
            max_iter=1000,
            random_state=20260824,
        ),
    ),
])

logistic.fit(
    train[features],
    train[TARGET],
)

logistic_valid = logistic.predict_proba(
    valid[features]
)[:, 1]

logistic_test = logistic.predict_proba(
    test[features]
)[:, 1]

logistic_result = {
    "model": "logistic_regression",
    "validation": evaluate(
        valid[TARGET],
        logistic_valid,
    ),
    "test": evaluate(
        test[TARGET],
        logistic_test,
    ),
}

print(logistic_result)


{'model': 'logistic_regression', 'validation': {'n_rows': 23471, 'positive_rate': np.float64(0.22994333432746794), 'roc_auc': 0.8725728501457033, 'pr_auc': 0.7108449642797692, 'log_loss': 0.35374703960445436, 'brier': 0.10913506082650717}, 'test': {'n_rows': 23567, 'positive_rate': np.float64(0.22590911019646115), 'roc_auc': 0.8810956503267332, 'pr_auc': 0.723008871212467, 'log_loss': 0.3417489142733216, 'brier': 0.10460433578227606}}


## 5. Gradient Boosting

In [6]:

gradient = Pipeline([
    ("preprocess", make_preprocessor()),
    (
        "model",
        GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=3,
            min_samples_leaf=50,
            subsample=0.85,
            random_state=20260824,
        ),
    ),
])

gradient.fit(
    train[features],
    train[TARGET],
)

gradient_valid = gradient.predict_proba(
    valid[features]
)[:, 1]

gradient_test = gradient.predict_proba(
    test[features]
)[:, 1]

gradient_result = {
    "model": "gradient_boosting",
    "validation": evaluate(
        valid[TARGET],
        gradient_valid,
    ),
    "test": evaluate(
        test[TARGET],
        gradient_test,
    ),
}

print(gradient_result)


{'model': 'gradient_boosting', 'validation': {'n_rows': 23471, 'positive_rate': np.float64(0.22994333432746794), 'roc_auc': 0.9109910825298151, 'pr_auc': 0.7746319158239297, 'log_loss': 0.3101993012577699, 'brier': 0.09481072046123482}, 'test': {'n_rows': 23567, 'positive_rate': np.float64(0.22590911019646115), 'roc_auc': 0.9149823653323921, 'pr_auc': 0.7873518476956868, 'log_loss': 0.3005952401651111, 'brier': 0.09089956540143809}}


## 6. Baseline Leaderboard

In [7]:

results = [
    base_result,
    logistic_result,
    gradient_result,
]

rows = []

for result in results:
    for split_name in ["validation", "test"]:
        row = {
            "model": result["model"],
            "split": split_name,
        }
        row.update(result[split_name])
        rows.append(row)

leaderboard = pd.DataFrame(rows)
display(
    leaderboard.sort_values(
        ["split", "roc_auc"],
        ascending=[True, False],
    )
)

leaderboard.to_csv(
    REPORT_DIR / "baseline_metrics.csv",
    index=False,
)

with open(
    REPORT_DIR / "baseline_results.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(results, f, indent=2)


,model,split,n_rows,positive_rate,roc_auc,pr_auc,log_loss,brier
5,gradient_boosting,test,23567,0.225909,0.914982,0.787352,0.300595,0.090900
3,logistic_regression,test,23567,0.225909,0.881096,0.723009,0.341749,0.104604
1,base_rate,test,23567,0.225909,0.500000,0.225909,0.534301,0.174880
4,gradient_boosting,validation,23471,0.229943,0.910991,0.774632,0.310199,0.094811
2,logistic_regression,validation,23471,0.229943,0.872573,0.710845,0.353747,0.109135
0,base_rate,validation,23471,0.229943,0.500000,0.229943,0.539216,0.177072


## 7. Interpretation Rules

Do not choose the final model from the test set.

Use validation performance to compare the fixed baselines.

The baseline questions are:

1. Does Logistic Regression beat the base-rate baseline?
2. Does Gradient Boosting improve on Logistic Regression?
3. Do ROC-AUC and PR-AUC agree?
4. Is probability quality reasonable from Log Loss/Brier?
5. Does performance remain sensible on the untouched test set?

C5 does not perform tuning, calibration, or policy/uplift evaluation yet.